# 06 · Behavioral feature investigation
## Round 1 — Measure representation value, not feature count

**280 candidate features · four families · 20 matched CPU fits · eight Plotly figures**

Primary question: do speech acts, attribution/scope, rule–behavior interactions and cross-fitted support contrasts add to the same lexical model? This is an exploratory original-data ablation, **not a new Kaggle score or a promotion of the accepted Qwen model**.

The run uses your existing locked environment and raw training file. No download, neural inference, cloud API call, automatic submission, or change to GitHub occurs.

### How to use
Run cells in order, or use **Run → Run All Cells** once after the installer reports `FEATURE_ROUND_INSTALLED`. The computational cell launches one bounded local worker and waits for it; it is not a background job. Completed candidates are checksum-verified and reused. On any error, stop and preserve the outputs. When using the terminal’s `--run` option, this notebook is already executed; do not run it again.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/behavioral_features.json").exists()), Path.home() / "projects/jigsaw-rule-classifier")
assert (ROOT / "configs/behavioral_features.json").is_file(), "Open this notebook inside the Jigsaw project"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from scripts.run_behavioral_features import bounded_compute, figures, write_dashboard
CONFIG = json.loads((ROOT / "configs/behavioral_features.json").read_text())
print("Project:", ROOT)
print("Hypothesis:", CONFIG["primary"])
print("Scientific worker limit:", CONFIG["max_seconds"], "seconds")

Project: /home/sagemaker-user/projects/jigsaw-rule-classifier
Hypothesis: full vs lexical_control
Scientific worker limit: 240 seconds


## 1. Research rationale and availability

Community-norm work motivates policy-sensitive representations rather than toxicity alone. Rule By Example motivates exemplar/rule grounding; CheckList motivates explicit tests of linguistic behavior. These sources **do not prove these heuristic features will help Jigsaw**. The winning competition system also used fine-tuning and ensembling, so we do not assume the remaining gap is entirely features.

[NormVio](https://aclanthology.org/2021.findings-emnlp.288/) · [Rule By Example](https://aclanthology.org/2023.acl-long.22/) · [CheckList](https://aclanthology.org/2020.acl-main.442/)

All features use comment/rule text and explicitly labeled supplied support examples. No invented context, target-derived identifiers, cross-rule negative labels, or hidden targets. Requests **and** offers may both be prohibited; a disclaimer or quotation never automatically defines a label.

In [2]:
family_table = pd.DataFrame([
    ["Behavior", 58, "Speech-act/action cues and counts"],
    ["Scope", 121, "Author/quote/code distinctions and approximate negation"],
    ["Rule alignment", 36, "Rule-mentioned behavior and sentence co-occurrence"],
    ["Support contrast", 65, "Inner-group-cross-fitted same-rule comparisons"],
], columns=["Family", "Candidates", "Hypothesis"])
display(family_table)
print("Total new dense candidates:", int(family_table.Candidates.sum()))
print("Per-family training-only selection cap:", CONFIG["feature_budget_per_family"])

,Family,Candidates,Hypothesis
0,Behavior,58,Speech-act/action cues and counts
1,Scope,121,Author/quote/code distinctions and approximate...
2,Rule alignment,36,Rule-mentioned behavior and sentence co-occurr...
3,Support contrast,65,Inner-group-cross-fitted same-rule comparisons


Total new dense candidates: 280
Per-family training-only selection cap: 24


## 2. Run the fixed ablations

The canonical support-adaptation plan excludes known support text globally, reproducing 234 advertising + 647 legal-advice novel queries. Notebook 05 used a narrower per-policy support lookup and reported one more eligible legal-advice comment. The study preserves that earlier report and displays the distinction rather than changing cohort silently.

Training supports are permitted inputs of the new rule; this is **not zero-shot evaluation**. Inner training features are cross-fitted by normalized body so a training row cannot act as its own labeled support.

The worker is capped at 240 seconds; a separate 300-second process-tree timeout bounds the notebook call. Watch the 15-second heartbeat and 1/20…20/20 counters. It uses one fixed logistic-regression model specification for each representation, not a hyperparameter sweep.

In [3]:
result = bounded_compute(ROOT)
print("Study status:", result["status"])
print("Research decision:", result["decision"])
print("Run ID:", result["run_id"])
print("Fits in the completed study:", result["fit_count"])
print("New neural forward passes:", result["new_neural_inference"])
display(pd.DataFrame(result["cohorts"]).drop(columns=["query_identity"]))

{"timestamp": "2026-09-11T22:14:38+00:00", "stage": "behavioral_feature_round1", "event": "started", "elapsed_seconds": 0.0, "stage_elapsed_seconds": 0.0, "total_elapsed_seconds": 0.0}


{"timestamp": "2026-09-11T22:14:39+00:00", "stage": "behavioral_feature_round1", "event": "cohort_verified", "elapsed_seconds": 0.829, "stage_elapsed_seconds": 0.829, "total_elapsed_seconds": 0.829, "queries": 234, "training_pairs": 1640}
{"timestamp": "2026-09-11T22:14:39+00:00", "stage": "behavioral_feature_round1", "event": "cohort_verified", "elapsed_seconds": 0.829, "stage_elapsed_seconds": 0.829, "total_elapsed_seconds": 0.829, "queries": 647, "training_pairs": 1215}


{"timestamp": "2026-09-11T22:14:47+00:00", "stage": "behavioral_feature_round1", "event": "candidate_complete", "elapsed_seconds": 9.44, "stage_elapsed_seconds": 9.44, "total_elapsed_seconds": 9.44, "fold": 0, "variant": "lexical_control", "completed": 1, "total": 20, "new_fits": 1, "reused_fits": 0}


{"timestamp": "2026-09-11T22:14:48+00:00", "stage": "behavioral_feature_round1", "event": "candidate_complete", "elapsed_seconds": 9.771, "stage_elapsed_seconds": 9.771, "total_elapsed_seconds": 9.771, "fold": 0, "variant": "add_behavior", "completed": 2, "total": 20, "new_fits": 2, "reused_fits": 0}


{"timestamp": "2026-09-11T22:14:48+00:00", "stage": "behavioral_feature_round1", "event": "candidate_complete", "elapsed_seconds": 10.095, "stage_elapsed_seconds": 10.095, "total_elapsed_seconds": 10.095, "fold": 0, "variant": "add_scope", "completed": 3, "total": 20, "new_fits": 3, "reused_fits": 0}


...

{"timestamp": "2026-09-11T22:15:01+00:00", "stage": "behavioral_feature_round1", "event": "candidate_complete", "elapsed_seconds": 23.139, "stage_elapsed_seconds": 23.139, "total_elapsed_seconds": 23.139, "fold": 1, "variant": "without_rule_alignment", "completed": 19, "total": 20, "new_fits": 19, "reused_fits": 0}


{"timestamp": "2026-09-11T22:15:02+00:00", "stage": "behavioral_feature_round1", "event": "candidate_complete", "elapsed_seconds": 23.531, "stage_elapsed_seconds": 23.531, "total_elapsed_seconds": 23.531, "fold": 1, "variant": "without_support_contrast", "completed": 20, "total": 20, "new_fits": 20, "reused_fits": 0}


{"timestamp": "2026-09-11T22:15:02+00:00", "stage": "behavioral_feature_round1", "event": "results_saved", "elapsed_seconds": 23.822, "stage_elapsed_seconds": 23.822, "total_elapsed_seconds": 23.822, "decision": "DO_NOT_PROMOTE_THIS_FEATURE_SET", "macro_delta": -0.010553495608784047}
{"timestamp": "2026-09-11T22:15:02+00:00", "stage": "behavioral_feature_round1", "event": "completed", "elapsed_seconds": 23.822, "stage_elapsed_seconds": 23.822, "total_elapsed_seconds": 23.822, "error_type": null}
RESULT: FEATURE_ROUND_COMPLETE
DECISION: DO_NOT_PROMOTE_THIS_FEATURE_SET


Study status: FEATURE_ROUND_COMPLETE
Research decision: DO_NOT_PROMOTE_THIS_FEATURE_SET
Run ID: 32e706fe1e7dddb7de34
Fits in the completed study: 20
New neural forward passes: 0


,policy,readiness_same_rule_novel,canonical_global_novel,cross_rule_support_exclusions,training_pairs,epoch_weight,query_text_in_training
0,"No Advertising: Spam, referral links, unsolici...",234,234,0,1640,2269,0
1,No legal advice: Do not offer or request legal...,648,647,1,1215,1581,0


## 3. Official metric family and secondary diagnostics

The figures below show local ROC AUC for each policy, and secondary Brier/log-loss diagnostics. They are not a Kaggle submission score. The fixed CPU control is not your accepted Qwen model. The primary comparison was fixed as the full feature set versus lexical control before these new scores were observed.

In [4]:
display(pd.DataFrame(result["metrics"]))
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS = figures(result)
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
2,0,"No Advertising: Spam, referral links, unsolici...",add_scope,0.661082,0.238971,0.734341,234
3,0,"No Advertising: Spam, referral links, unsolici...",add_rule_alignment,0.624366,0.261458,0.765684,234
4,0,"No Advertising: Spam, referral links, unsolici...",add_support_contrast,0.629515,0.260484,0.743546,234
5,0,"No Advertising: Spam, referral links, unsolici...",full,0.629440,0.263447,0.786720,234
6,0,"No Advertising: Spam, referral links, unsolici...",without_behavior,0.625634,0.268217,0.797409,234
7,0,"No Advertising: Spam, referral links, unsolici...",without_scope,0.628694,0.263784,0.780773,234
8,0,"No Advertising: Spam, referral links, unsolici...",without_rule_alignment,0.632127,0.259566,0.792461,234
9,0,"No Advertising: Spam, referral links, unsolici...",without_support_contrast,0.635784,0.253998,0.757287,234


,variant,pooled_auc,ranked_pooled_auc,rows
0,lexical_control,0.641345,0.650247,881
1,add_behavior,0.680777,0.681784,881
2,add_scope,0.673588,0.674928,881
3,add_rule_alignment,0.665208,0.668093,881
4,add_support_contrast,0.664349,0.666817,881
5,full,0.654366,0.655178,881
6,without_behavior,0.656385,0.657451,881
7,without_scope,0.659275,0.660984,881
8,without_rule_alignment,0.657097,0.657472,881
9,without_support_contrast,0.667381,0.667592,881


## 4. Paired uncertainty and actual family contributions

The 500 comment-group bootstrap draws are paired across all outputs. The simultaneous intervals cover the 13 registered macro-AUC contrasts, not the full adaptive project history. They condition on these fitted predictions and two repeatedly inspected policies; they are not independent confirmation.

A feature family must earn its place through addition/removal evidence—not a coefficient or a large column count.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")
CHARTS[2].show(renderer="plotly_mimetype")

,comparison,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,add_behavior vs control,0.022354,-0.016498,0.061205,500
1,add_scope vs control,0.013012,-0.025840,0.051864,500
2,add_rule_alignment vs control,-0.003336,-0.042187,0.035516,500
3,add_support_contrast vs control,-0.002571,-0.041423,0.036281,500
4,full vs control,-0.010553,-0.049405,0.028298,500
5,without_behavior vs control,-0.010196,-0.049048,0.028655,500
6,without_scope vs control,-0.006788,-0.045639,0.032064,500
7,without_rule_alignment vs control,-0.008114,-0.046966,0.030737,500
8,without_support_contrast vs control,-0.000058,-0.038910,0.038794,500
9,ablation: behavior,-0.000357,-0.039209,0.038494,500


## 5. Training-only screening and fold stability

These plots show how many candidate columns were kept and whether their names persisted across folds. Constant and exact-duplicate columns are rejected. Stability is informative but not evidence of predictive value by itself.

In [6]:
display(pd.DataFrame(result["selection"]).drop(columns=["names"]))
CHARTS[3].show(renderer="plotly_mimetype")
CHARTS[4].show(renderer="plotly_mimetype")

,fold,policy,family,candidates,constant,retained
0,0,"No Advertising: Spam, referral links, unsolici...",behavior,58,0,24
1,0,"No Advertising: Spam, referral links, unsolici...",scope,121,46,24
2,0,"No Advertising: Spam, referral links, unsolici...",rule_alignment,36,16,19
3,0,"No Advertising: Spam, referral links, unsolici...",support_contrast,65,14,24
4,1,No legal advice: Do not offer or request legal...,behavior,58,2,24
5,1,No legal advice: Do not offer or request legal...,scope,121,50,24
6,1,No legal advice: Do not offer or request legal...,rule_alignment,36,16,19
7,1,No legal advice: Do not offer or request legal...,support_contrast,65,16,24


## 6. Attribution diagnostics and cohort reconciliation

Standardized dense coefficients are associations under this fitted linear readout. They are not causal effects and do not replace ablations. The cohort chart documents the global versus within-policy support exclusion. Neither the earlier readiness statistic nor the historical comparison is silently rewritten.

In [7]:
CHARTS[5].show(renderer="plotly_mimetype")
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

## 7. Decision — stop, inspect, or justify one next experiment

The full candidate must gain at least +0.003 macro AUC, have a simultaneous lower bound above zero, and have no policy regression, and not reduce local per-policy-ranked pooled AUC. `DO_NOT_PROMOTE_THIS_FEATURE_SET` is a **valid completed negative result**, not a script error. `ELIGIBLE_FOR_NEXT_VALIDATION_ONLY` is not permission to claim a record, run a GPU sweep, or replace the accepted model.

No feature family is declared exhausted by one negative screen. Continue only with a new, specific hypothesis and evidence for the compute cost.

In [8]:
display(pd.DataFrame([result["primary_comparison"]]))
print("Per-policy primary deltas:", result["primary_policy_deltas"])
print("Decision:", result["decision"])
print("Automatic GPU authorization:", result["automatic_gpu_authorization"])
for limitation in result["limitations"]:
    print("•", limitation)
DASHBOARD = write_dashboard(ROOT, result)
print("Interactive dashboard:", DASHBOARD)
print("Private checkpoints:", ROOT / "runs/behavioral_features" / result["run_id"])
print("Save this notebook, then use the package export command. Do not upload private checkpoints.")

,comparison,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,full vs control,-0.010553,-0.049405,0.028298,500


Per-policy primary deltas: [-0.043582089552238745, 0.02247509833467065]
Decision: DO_NOT_PROMOTE_THIS_FEATURE_SET
Automatic GPU authorization: False
• The original-data development set has been inspected repeatedly; it is not a fresh holdout.
• Fixed lexical CPU readout tests feature value; it does not measure an improvement over the accepted Qwen model.
• Heuristic speech-act, quotation and negation cues are fallible; their firing never supplies a label.
• Bootstrap intervals are conditional on these fitted predictions, not independent policy or retraining uncertainty.
• No external threads, timestamps, unseen labels or cross-rule negative labels were manufactured.
• A negative finding rejects this registered representation, not all feature engineering.


Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/behavioral_features/dashboard.html
Private checkpoints: /home/sagemaker-user/projects/jigsaw-rule-classifier/runs/behavioral_features/32e706fe1e7dddb7de34
Save this notebook, then use the package export command. Do not upload private checkpoints.
